## Hälsostudie
**Student:** Wendy Mermet

**Datakälla:** "health_study_dataset.csv.csv"


# Sammanfattning av hälsostudiedata

Vi har analyserat hälsostudiedata med beskrivande statistik för ålder, vikt, längd, systoliskt blodtryck och kolesterol, 
inklusive medelvärde, median samt min- och maxvärden. Vi skapade en dashboard med histogram över blodtryck, 
boxplots över vikt per kön och stapeldiagram över andelen rökare, vilket gav tydlig visuell insikt i datafördelningarna. 

En simulering av sjukdomsförekomst visade att andelen sjuka i 1 000 påhittade deltagare stämde väl överens med verklig 
andel i datasetet, vilket bekräftar sannolikhetsmönstret. 

Hypotesprövning visade ingen statistiskt signifikant skillnad i 
medelblodtryck mellan rökare och icke-rökare (p = 0,6854, 95% CI: -1,636 till 2,465), med mycket liten effektstorlek 
(Cohen’s d = 0,037).

Regressionsanalysen för blodtryck gav modellen:

    Blodtryck = 109,499 + 0,539 * Ålder + 0,178 * Vikt,  R² = 0,405
    
vilket visar att både ålder och vikt är positiva prediktorer.

### Rekommendationer för nästa steg

1. **Öka datamängden:** Inkludera fler deltagare för att öka statistisk styrka och kunna upptäcka små effektstorlekar.
2. **Inkludera fler variabler:** Faktorer som fysisk aktivitet, kost, genetik eller medicinering kan bidra till variation i blodtryck och bör övervägas.
3. **Utför fler hypotesprövningar:** Analysera skillnader mellan grupper, t.ex. kön eller åldersgrupper, för att identifiera fler relevanta samband.
4. **Förbättra prediktionsmodellen:** Överväg mer avancerade modeller (multipel regression med interaktioner eller maskininlärning) för att bättre förklara variationen i blodtryck.
5. **Fortsatt visualisering och dashboardutveckling:** Använd interaktiva funktioner för att utforska data och identifiera mönster över tid eller mellan subgrupper.

Sammantaget ger arbetet en tydlig bild av data, identifierar centrala samband och lägger en solid grund för vidare analyser och beslut.



**Sammanfattning av medelvärde, median, min och max för:**
- ålder
- vikt
- längd
- systoliskt blodtryck
- kolesterol



In [ ]:
import src.HealthAnalyzeTools as hat
import src.visuals as vis

analysis = hat.HealthAnalyzer()
analysis.main_summary()


**Dashboard som visar följande grafer från hälsostudiedata**
- Histogram över blodtryck
- Boxplot över vikt per kön
- Stapeldiagram över andelen rökare

*Jag använde hjälp från chatten för att skapa Dashboard-funktionen och modifiera graf-funktionerna så att Dashboard-displayen blev korrekt.*


In [ ]:
g1 = analysis.plot_blood_pressure_histogram() # histogram över blodtryck
g2 = analysis.plot_weight_boxplot() # boxplot över vikt per kön
g3 = analysis.plot_smoker_bar_chart() #stapeldiagram över andelen rökare


vis.add_to_dashboard([g1, g2, g3],figsize=(13,8), title="Hälsostudie Översikt")

Vi gör en enkel simulering för att förstå hur vanligt det är att deltagare i datasetet är sjuka.

1. **Räkna ut verklig andel:**  
   Först tar vi reda på hur stor del av personerna i det verkliga materialet som är sjuka.

2. **Skapa en simulerad grupp:**  
   När vi vet den verkliga andelen skapar vi sedan 1 000 påhittade personer.  
   Var och en av dessa får samma chans att vara sjuk som den verkliga andelen vi räknade fram.

3. **Jämför verklighet och simulering:**  
   När simuleringen är klar jämför vi hur många av de 1 000 påhittade personerna som är sjuka med hur många som faktiskt var sjuka i verkligheten.

Detta hjälper oss att se hur väl simuleringen stämmer överens med det riktiga datamaterialet.

In [ ]:
stats = hat.HealthStatistics()

print(f"~{stats.perc_sick_participants():.3f}% of the participants declare having a disease")

mean_diseases, mean_sick_df = stats.simulate_disease()
print(f"The average of sick participants in the simulated data base of 1 000 is {mean_diseases:.3f}%\nThe average of sick participants in the original data base is {mean_sick_df:.3f}%")


Det faktum att simuleringen och det verkliga utfallet ligger så nära varandra visar att era riktiga data följer det sannolikhetsmönster som vi använde i simuleringen. Det betyder att simuleringen stämmer bra överens med det riktiga datamaterialet, vilket var målet i detta steg.

Nu beräknar vi ett **konfidensintervall för medelvärdet av systolic_bp** med hjälp av två olika metoder:
- med normalapproximation
- med bootstrap method


In [ ]:
n_lo, n_hi, n_mean_x, _, _ = stats.ci_mean_normal(stats.x)
print(f"Using the normal approximation method:\nthe 95% confidence intervall for the average of the blood pressure is {n_lo:.3f} -{n_hi:.3f} mmHg. ")
g5 =stats.ci_mean_normal_graph(n_lo, n_hi, n_mean_x)

b_lo, b_hi, b_mean_x = stats.ci_mean_bootstrap(stats.x)
print(f"Using the bootstrap method:\nthe 95% confidence intervall for the average of the blood pressure is {b_lo:.3f} -{b_hi:.3f} mmHg. ")
g6 =stats.ci_mean_boot_graph(b_lo, b_hi, b_mean_x)

vis.add_to_dashboard([g5,g6],figsize=(13,9), title="95 % Konfidensintervall för medelvärde blodtryck")

### Hypotesprövning

Vi testade hypotesen **"Rökare har högre medel-blodtryck än icke-rökare"** med hjälp av bootstrop metod. 
Bootstrap-analysen gav följande resultat:

- **p-värde:** 0.6854  
- **95% konfidensintervall för skillnaden i medelvärden:** -1.636 till 2.465  

**Tolkning:**  
Eftersom p-värdet är betydligt högre än 0.05 och konfidensintervallet innehåller 0 finns det **ingen statistiskt signifikant skillnad** i medel-blodtryck mellan rökare och icke-rökare i detta dataunderlag.

**Slutsats:** Resultaten ger **inte stöd** för hypotesen att rökare har högre medel-blodtryck än icke-rökare.



In [ ]:
p_boot, cilow, cihigh, true_diff = stats.hypothesis_test_smoker_bp()

print(f"Bootstrap method: p-value = {p_boot:.4f}, true difference in means = {true_diff:.3f}")
print(f"95% CI for the difference in means: {cilow:.3f} to {cihigh:.3f}")


In [ ]:
d, n_per_group = stats.cohen_d_smoker_bp()

print(f"Cohen's d effect size: {d:.3f}")
print(f"Sample size per group: {n_per_group}")

import statsmodels.stats.power as power_analysis

power_analysis = power_analysis.TTestIndPower()
power_t = power_analysis.solve_power(effect_size=d, nobs1=n_per_group, alpha=0.05, ratio=1.0, alternative='larger')
print(f"Calculated power of the test: {power_t:.3f}")

n_needed = power_analysis.solve_power(effect_size=d, power=0.8, alpha=0.05, ratio=1.0, alternative='larger')
print(f"Required sample size per group for 80% power: {n_needed:.0f}")

print( f"""
The observed effect size (Cohen's d = {power_t:.3f}) is extremely small.
With a sample size of {n_per_group} per group, the statistical power is only {power_t*100:.1f}%, indicating a {100 - (power_t * 100):.1f}% chance of failing to detect the effect.
To achieve 80% power for such a small effect, approximately {n_needed:.0f} participants per group would be required. 
Therefore, the current study is underpowered and the detected effect is likely negligible in practical terms.
""")

# Enkel linjär regression för att förutsäga blodtryck från ålder och vikt.


In [ ]:
intercept, coef_age, coef_weight, r_squared, predictions = stats.linear_regression_age_weight_bp()

print("Linear Regression Model Coefficients:")
print(f"Intercept: {intercept:.3f}")
print(f"Age Coefficient: {coef_age:.3f}")
print(f"Weight Coefficient: {coef_weight:.3f}") 
print(f"R-squared: {r_squared:.3f}")

corr_age_bp, corr_weight_bp = stats.corr_age_weight_bp()
print()
print("Correlation between age and systolic blood pressure:")
print(f"{corr_age_bp:.3f}") 
print("Correlation between weight and systolic blood pressure:")
print(f"{corr_weight_bp:.3f}")

g7 = stats.plot_regression_age_bp(predictions=predictions)
g8 = stats.plot_regression_weight_bp(predictions=predictions)
g9 = stats.plot_predictions_bp(predictions=predictions)
vis.add_to_dashboard([g7,g8,g9],figsize=(13,13), title="Linjära regressioner av blodtryck mot ålder och vikt")


# Regressionsmodell
***Blodtryck = 109.499 + 0.539 * Ålder + 0.178 * Vikt***

**Tolkning av koefficienter**
- <u>Intercept:</u> ***109.499***

    Detta är det förväntade blodtryck värdet när ålder = 0 och vikt = 0.
    
- <u>Ålderskoefficient:</u> ***0.539***

    För varje 1-års ökning i ålder förutsäger modellen att blodtryck ökar med 0.539 mmHg, givet att vikten hålls konstant.
    Exempel: Om åldern ökar från 30 till 31 ökar blodtryck med ungefär 0.539 mmHg

- <u>Viktkoefficient:</u> ***0.178***

    För varje 1kg ökning i vikt förutsäger modellen att blodtryck ökar med 0.178 mmHg, givet att åldern hålls konstant.
    Exempel: Om vikten ökar med 10kg ökar blodtryck med 1.78 mmHg.

- <u>Modellens förklaringsgrad:</u> ***R² = 0.405***

    Detta innebär att modellen förklarar 40.5 % av variationen i utfallsvariabeln.

**Sammanfattning**

Ålder och vikt har båda positiva samband med utfallet.
Ålder har en starkare effekt än vikt (0.539 vs. 0.178).
Modellen har en måttligt god passform, men skulle kunna förbättras genom att inkludera fler variabler.